# Dialogue Summarizer — Training on Colab T4

Fine-tunes `microsoft/Phi-3-mini-4k-instruct` on DialogSum (`knkarthick/dialogsum`) using LoRA (PEFT).

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Add your HuggingFace token in the Colab Secrets tab (key icon, name: `HF_TOKEN`)

**Expected runtime:** ~1–2 hours for 3 epochs on ~12k examples.

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found! Set Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
# Colab already has torch, so we skip it to avoid version conflicts
!pip install -q \
    "datasets>=2.0.0" \
    "transformers>=4.40.0" \
    "peft>=0.19.0" \
    "bitsandbytes>=0.43.0" \
    "accelerate>=0.30.0" \
    "rouge-score==0.1.2" \
    "mlflow>=2.13.0" \
    "gradio>=4.31.5" \
    "python-dotenv==1.0.1"
print("Dependencies installed.")

## 3. Clone the repo

In [ ]:
import os

REPO_URL = "https://github.com/rotemso23/samsum-phi3-lora.git"  # update if repo name differs
REPO_DIR = "samsum-phi3-lora"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

## 4. Set HuggingFace token

Your token needs **read + write** permissions.  
Get one at: https://huggingface.co/settings/tokens

In [ ]:
from google.colab import userdata
import os

# Option A: read from Colab Secrets (Secrets tab on the left sidebar → add HF_TOKEN)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    # Option B: paste directly (don't commit this)
    os.environ["HF_TOKEN"] = "hf_xxx_YOUR_TOKEN_HERE"
    print("HF_TOKEN set manually — remember not to commit this notebook with a real token.")

# Write to .env so train.py can find it via python-dotenv
with open(".env", "w") as f:
    f.write(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
print("Token written to .env")

## 5. Sanity-check data pipeline

In [ ]:
!PYTHONPATH=/content/samsum-phi3-lora python src/data.py

## 6. Train

This takes **~1–2 hours** on a T4. Watch for:
- Train loss dropping across steps (good sign)
- Eval loss after each epoch (should also drop)
- OOM error → reduce `per_device_train_batch_size` to 2 in `src/train.py`

In [ ]:
!PYTHONPATH=/content/samsum-phi3-lora python src/train.py

## 7. Check the model on Hub

After training completes, the adapter is published at:  
https://huggingface.co/rotemso23/samsum-phi3-lora

In [ ]:
import sys
sys.path.insert(0, "/content/samsum-phi3-lora")

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

HUB_REPO = "rotemso23/samsum-phi3-lora"
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(HUB_REPO, trust_remote_code=True)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
model = PeftModel.from_pretrained(base, HUB_REPO)
model.eval()

dialogue = """Amanda: I baked cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you some tomorrow."""

from src.data import INSTRUCTION
messages = [{"role": "user", "content": f"{INSTRUCTION}\n\nConversation:\n{dialogue}"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=128, do_sample=False)

generated = output[0][inputs["input_ids"].shape[-1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

## 8. (Optional) View MLflow experiment

MLflow logs are saved to `mlflow_runs/`. To view them after training:  
Download the `mlflow_runs/` folder and run locally:
```bash
mlflow ui --backend-store-uri mlflow_runs
```
Then open http://localhost:5000